<a href="https://colab.research.google.com/github/444112029012/phishing-detection-project/blob/main/colab/%E5%89%B5%E5%BB%BA%E8%B3%87%E6%96%99%E9%9B%86/HTML_play_wright.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install playwright
!playwright install chromium
!playwright install-deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('phishing_dataset_html_combine2_html.csv')
df.head()

,url,target,phish_hints,domain_in_brand,nb_hyperlinks,ratio_intHyperlinks,ratio_extHyperlinks,ratio_extRedirection,ratio_extErrors,external_favicon,links_in_tags,ratio_extMedia,safe_anchor,empty_title,domain_in_title,domain_with_copyright,has_meta_refresh,has_js_redirect,feature_extracted
0,http://www.ishanarora.com/2009/07/29/windows-7...,0,0,0,29,0.827586,0.172414,0.0,0.000000,0,51,0.0,0,0,0,0,0,0,1
1,http://icyte.com/snapshots/show/05deeb7f5c0c21...,1,0,1,0,0.000000,0.000000,0.0,0.000000,0,4,0.0,1,0,0,0,0,0,1
2,http://ustasb.com/officesnake/,0,0,0,1,0.000000,1.000000,0.0,0.000000,0,10,0.0,1,0,0,0,0,0,1
3,http://www.dw.com/en/us-and-somali-forces-stri...,0,0,1,74,0.648649,0.351351,0.0,0.027027,0,103,0.0,0,0,0,1,0,0,1
4,https://tinyurl.com/sdi-template,1,0,1,24,0.958333,0.041667,0.0,0.041667,0,49,0.0,0,0,0,0,0,0,1


In [ ]:
df = df[['url', 'target']]

In [ ]:
df.to_csv('phishing_dataset_html_combine2_html.csv', index=False)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140978 entries, 0 to 140977
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     140978 non-null  object
 1   target  140978 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.2+ MB


# **批次處理**

In [ ]:
import pandas as pd
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import re
import asyncio
import os
import numpy as np
import gc
import ipaddress
from typing import Tuple, Optional
from google.colab import drive

# ====== Playwright Async 模組 ======
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError, Error as PlaywrightError

# --- 📁 第一招：Google Drive 掛載與資料夾設定 ---
drive.mount('/content/drive')

FILE_NAME = "/content/drive/MyDrive/畢業專題Playwright_html/phishing_dataset_html_combine2_html.csv"

# --- ⚙️ 第二招：併發與批次設定 ---
CONCURRENCY_LIMIT = 20   # 同時開啟的分頁數 (Colab 免費版建議 5~10)
BATCH_SIZE = 200        # 每處理幾筆存一次檔
RESTART_INTERVAL = 200  # 每處理幾筆強制重啟瀏覽器釋放記憶體
RENDER_WAIT_TIME = 2

# 建立號誌，限制同時執行的任務數量
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

async def block_agressive_resources(route):
    """阻擋圖片、影片、CSS 載入，大幅提升速度"""
    if route.request.resource_type in ["image", "media", "font", "stylesheet"]:
        await route.abort()
    else:
        await route.continue_()

async def get_html_content_async(session, url, timeout=20, max_retries=2):
    """使用 aiohttp 快速獲取靜態網頁"""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.7258.66 Safari/537.36'}
    for attempt in range(max_retries):
        try:
            async with session.get(url, headers=headers, timeout=timeout, ssl=False) as response:
                if response.status >= 400:
                    return None, False

                html_content = await response.text()
                # 第三招：改用 lxml 加速解析
                soup = BeautifulSoup(html_content, 'lxml')
                body_content = soup.find('body')

                if html_content and len(html_content.strip()) > 100:
                    not_found_keywords = ['page not found', 'error 404', 'page does not exist', '找不到頁面', '頁面不存在']
                    text_to_check = body_content.get_text(strip=True).lower() if body_content else html_content.lower()
                    matched_keywords = sum(1 for kw in not_found_keywords if kw in text_to_check)

                    if matched_keywords >= 2:
                        return None, False
                    return html_content, False
                else:
                    break
        except Exception:
            if attempt == max_retries - 1: break
            await asyncio.sleep(1)
    return None, None

async def fetch_dynamic_content_async(page, url: str) -> Tuple[Optional[str], str]:
    """使用 Playwright 獲取動態網頁 (包含轉址處理)"""
    try:
        try:
            response = await page.goto(url, timeout=20000, wait_until='domcontentloaded')
        except PlaywrightError as e:
            if "interrupted by another navigation" in str(e):
                try: await page.wait_for_load_state('domcontentloaded', timeout=20000)
                except: pass
            else:
                raise e

        await page.wait_for_timeout(RENDER_WAIT_TIME * 1000)

        # 簡單滾動一下
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2);")
        await page.wait_for_timeout(500)

        text = await page.content()
        return (text, 'OK_Dynamic') if text else (None, 'OK_Dynamic_Empty')

    except Exception:
        return None, 'Error_Playwright'

async def process_single_row(index, row, session, context, df, html_feature_columns, total_rows):
  """處理單筆資料的工作節點 (Worker)"""
  url = row['url']
  if pd.isna(url) or not str(url).strip():
    df.at[index, 'feature_extracted'] = 0.0
    return

  if not url.startswith('http://') and not url.startswith('https://'):
    url = 'http://' + url

  # 每個任務獨立開一個 Page，避免互相干擾
  page = await context.new_page()
  await page.route("**/*", block_agressive_resources)

  try:
    print(f"🔍 處理中 [{index+1}/{total_rows}]: {url}")

    html_content, _ = await get_html_content_async(session, url, timeout=15, max_retries=1)
    dynamic_content = None
    if html_content:
      dynamic_content, _ = await fetch_dynamic_content_async(page, url)

    if html_content and dynamic_content:
      # ====== 以下是你的特徵解析邏輯 ======
      # 第三招：統一使用 'lxml' 加速
      soup_static = BeautifulSoup(html_content, 'lxml')
      soup_dynamic = BeautifulSoup(dynamic_content, 'lxml')

      parsed_url = urlparse(url)
      base_domain = parsed_url.netloc.split(':')[0]
      if base_domain.startswith('www.'): base_domain = base_domain[4:]

      # 15 & 16
      meta_tag = soup_static.find('meta', attrs={'http-equiv': lambda x: x and x.lower() == 'refresh'})
      df.at[index, 'has_meta_refresh'] = 1.0 if meta_tag and "url=" in meta_tag.get("content", "").lower() else 0.0
      redirect_kws = ["window.location.href", "location.href", "location.assign", "location.replace"]
      df.at[index, 'has_js_redirect'] = 1.0 if soup_static.find("script", string=lambda s: any(k in s for k in redirect_kws) if s else False) else 0.0

      # 1
      phish_kws = ['login', 'signin', 'account update', 'verify account', 'security alert', 'password', 'bank', 'paypal']
      text_content = soup_dynamic.get_text().lower()
      df.at[index, 'phish_hints'] = 1.0 if any(kw in text_content for kw in phish_kws) else 0.0

      # 2
      domain_parts = base_domain.split('.')
      core_domain = domain_parts[-2] if len(domain_parts) >= 2 and domain_parts[-1] in ['com', 'org', 'net', 'edu', 'gov'] else (domain_parts[-3] if len(domain_parts) >= 3 else domain_parts[0])
      title_tag = soup_dynamic.find('title')
      meta_desc = soup_dynamic.find('meta', attrs={'name': 'description'})
      df.at[index, 'domain_in_brand'] = 1.0 if (title_tag and core_domain in title_tag.get_text().lower()) or (meta_desc and core_domain in meta_desc.get('content', '').lower()) else 0.0

      # 3, 4, 5, 6, 7, 11
      all_links = soup_dynamic.find_all('a', href=True)
      df.at[index, 'nb_hyperlinks'] = len(all_links)

      int_links, ext_links, red_count, err_count = 0, 0, 0, 0
      is_safe_anchor = 1.0
      suspicious_kws = ['bit.ly', 'tinyurl', 'goo.gl', 't.co']

      for link in all_links:
          href = link['href']
          if href.startswith('#'): continue
          full_url = urljoin(url, href)
          linked_domain = urlparse(full_url).netloc
          if linked_domain == parsed_url.netloc:
            int_links += 1
          else:
            ext_links += 1
            if link.get('onclick') and 'window.location' in link.get('onclick', ''): red_count += 1
            elif link.get('target') == '_blank' and 'redirect' in link.get_text().lower(): red_count += 1
            if 'error' in full_url.lower() or '404' in full_url or not linked_domain: err_count += 1

            try: ipaddress.ip_address(linked_domain); is_safe_anchor = 0.0
            except ValueError: pass
            if any(kw in linked_domain.lower() for kw in suspicious_kws): is_safe_anchor = 0.0

      total_links = int_links + ext_links
      df.at[index, 'ratio_intHyperlinks'] = int_links / total_links if total_links > 0 else 0.0
      df.at[index, 'ratio_extHyperlinks'] = ext_links / total_links if total_links > 0 else 0.0
      df.at[index, 'ratio_extRedirection'] = red_count / len(all_links) if all_links else 0.0
      df.at[index, 'ratio_extErrors'] = err_count / len(all_links) if all_links else 0.0
      df.at[index, 'safe_anchor'] = is_safe_anchor

      # 8, 9, 10
      favicon = soup_dynamic.find('link', rel=lambda x: x and 'icon' in x.lower())
      df.at[index, 'external_favicon'] = 1.0 if favicon and 'href' in favicon.attrs and urlparse(urljoin(url, favicon['href'])).netloc != parsed_url.netloc else 0.0
      df.at[index, 'links_in_tags'] = sum(1 for tag in soup_dynamic.find_all(['a', 'script', 'img', 'link', 'iframe', 'form']) if 'href' in tag.attrs or 'src' in tag.attrs or (tag.name == 'form' and 'action' in tag.attrs))

      media_tags = soup_dynamic.find_all(['img', 'audio', 'video', 'source'])
      ext_media = sum(1 for tag in media_tags if (tag.get('src') or tag.get('href')) and urlparse(urljoin(url, tag.get('src') or tag.get('href'))).netloc != parsed_url.netloc)
      df.at[index, 'ratio_extMedia'] = ext_media / len(media_tags) if media_tags else 0.0

      # 12, 13, 14
      df.at[index, 'empty_title'] = 1.0 if not (title_tag and title_tag.string and title_tag.string.strip()) else 0.0
      df.at[index, 'domain_in_title'] = 1.0 if title_tag and title_tag.string and base_domain in title_tag.string.lower() else 0.0
      copyright_tags = soup_dynamic.find_all(text=re.compile(r'©|copyright|all rights reserved', re.IGNORECASE))
      df.at[index, 'domain_with_copyright'] = 1.0 if any(base_domain in tag.lower() for tag in copyright_tags) else 0.0

      df.at[index, 'feature_extracted'] = 1.0

    else:
      for col in html_feature_columns: df.at[index, col] = 0.0
      df.at[index, 'feature_extracted'] = 0.0

  except Exception as e:
    for col in html_feature_columns: df.at[index, col] = 0.0
    df.at[index, 'feature_extracted'] = 0.0
  finally:
    # ✅ 確保用完的 Page 一定會被關閉，釋放記憶體！
    await page.close()
async def safe_worker_wrapper(index, row, session, context, df, html_feature_columns, total_rows):
    # 先拿到「並發許可證（入座）」，才開始算 90 秒！
    async with semaphore:
        try:
            # 入座後，給他 90 秒的時間吃拉麵（爬蟲）
            await asyncio.wait_for(
                process_single_row(index, row, session, context, df, html_feature_columns, total_rows),
                timeout=90.0
            )
        except asyncio.TimeoutError:
            print(f"⏰ [上帝大限] 第 {index+1} 筆 URL ({row['url']}) 卡死超過 90 秒，已強制拔管！")
            df.at[index, 'feature_extracted'] = 0.0
            for col in html_feature_columns:
                if col != 'feature_extracted':
                    df.at[index, col] = 0.0
        except Exception as e:
            print(f"❌ [未預期崩潰] 第 {index+1} 筆 URL 發生錯誤: {e}")
            df.at[index, 'feature_extracted'] = 0.0

async def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
    html_feature_columns = [
        'phish_hints', 'domain_in_brand', 'nb_hyperlinks', 'ratio_intHyperlinks',
        'ratio_extHyperlinks', 'ratio_extRedirection', 'ratio_extErrors',
        'external_favicon', 'links_in_tags', 'ratio_extMedia', 'safe_anchor',
        'empty_title', 'domain_in_title', 'domain_with_copyright',
        'has_meta_refresh', 'has_js_redirect', 'feature_extracted'
    ]

    if 'feature_extracted' not in df.columns:
        df[html_feature_columns] = np.nan

    total_rows = len(df)
    print(f"總共 {total_rows} 筆資料準備進入併發處理...")

    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage'])
            context = await browser.new_context(ignore_https_errors=True)

            try:
                # 🔪 將任務切成 BATCH_SIZE 大小的批次
                for i in range(0, total_rows, BATCH_SIZE):
                    batch_df = df.iloc[i:i+BATCH_SIZE]
                    tasks = []

                    for index, row in batch_df.iterrows():
                        # 跳過已經成功提取特徵的資料 (斷點續傳)
                        # if index < 112000:
                        #   continue
                        if row.get('feature_extracted') == 1.0:
                            continue

                        # 建立併發任務
                        task = asyncio.create_task(
                            safe_worker_wrapper(index, row, session, context, df, html_feature_columns, total_rows)
                        )
                        tasks.append(task)

                    # 如果這個批次有任務需要跑，就一口氣執行它們
                    if tasks:
                        await asyncio.gather(*tasks, return_exceptions=True)
                    else:
                      continue
                    # 批次結束，進行存檔
                    print(f"\n💾 --- 已處理至 {min(i+BATCH_SIZE, total_rows)} 筆，寫入 Google Drive 中... ---")
                    df.to_csv(FILE_NAME, index=False)

                    # 記憶體回收機制
                    if i > 0 and i % RESTART_INTERVAL == 0:
                        print("♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...")
                        try: await context.close()
                        except: pass
                        try: await browser.close()
                        except: pass
                        gc.collect()
                        browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage'])
                        context = await browser.new_context(ignore_https_errors=True)

            except (KeyboardInterrupt, asyncio.CancelledError):
                print("\n🛑 偵測到手動中斷，儲存最終進度...")
                df.to_csv(FILE_NAME, index=False)
            finally:
                await browser.close()
    return df

async def main():
    os.system("pkill -9 -f chrome")

    if os.path.exists(FILE_NAME):
        print(f"✅ 找到 Drive 中的進度檔: {FILE_NAME}")
        df = pd.read_csv(FILE_NAME)
        # 如果你想「全部重來」，把下面這行取消註解：
        # df['feature_extracted'] = np.nan
    else:
        print(f"⚠️ 找不到進度檔。請確保原始 CSV 存在，或者修改讀取路徑。")
        # 這裡請替換成你的「最原始資料」的讀取路徑
        # df = pd.read_csv("/content/原本的原始檔.csv")
        return

    df_updated = await process_dataset(df)
    print("🎉 任務徹底完成！")

# 啟動任務
await main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 找到 Drive 中的進度檔: /content/drive/MyDrive/畢業專題Playwright_html/phishing_dataset_html_combine2_html.csv
總共 140978 筆資料準備進入併發處理...
🔍 處理中 [8/140978]: http://www.dictionarist.com/french-english/Sonar
🔍 處理中 [11/140978]: https://www.facebook.com/ImpactEventsGroupInc
🔍 處理中 [15/140978]: https://askubuntu.com/questions/434849/change-keyboard-layout-english-uk-on-command-line-to-english-us
🔍 處理中 [9/140978]: http://www.progarchives.com/album.asp?id=61737
🔍 處理中 [20/140978]: https://en.wikipedia.org/wiki/File_Transfer_Protocol
🔍 處理中 [27/140978]: http://www.imdb.com/title/tt2163227/
🔍 處理中 [32/140978]: https://shop.hostingdude.com/?q=http://take-away1.com/sms/olb
🔍 處理中 [22/140978]: http://military.wikia.com/wiki/United_States_Army_Pacific
🔍 處理中 [35/140978]: https://files.fm/u/k3z8xnt5
🔍 處理中 [41/140978]: https://www.tumblr.com/safe-mode?url=http%3A%2F%2Finceztum.tumblr.com%2F
🔍

/tmp/ipykernel_1531/3978108832.py:185: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  copyright_tags = soup_dynamic.find_all(text=re.compile(r'©|copyright|all rights reserved', re.IGNORECASE))



💾 --- 已處理至 400 筆，寫入 Google Drive 中... ---
♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...
🔍 處理中 [402/140978]: http://www.gothic.net/
🔍 處理中 [407/140978]: https://en.wikipedia.org/wiki/Food_processing
🔍 處理中 [408/140978]: http://howprotectmypass.webservis.ru
🔍 處理中 [420/140978]: https://en.wikipedia.org/wiki/Public_address_system
🔍 處理中 [416/140978]: https://www.baku.ru/
🔍 處理中 [421/140978]: http://list.ly/list/pXy-open-source-enterprise-architecture-modeling-tools
🔍 處理中 [426/140978]: https://www.quora.com/What-is-the-best-design-for-a-popsicle-stick-bridge
🔍 處理中 [409/140978]: http://www.english-for-students.com/Computer-Mouse.html
🔍 處理中 [423/140978]: https://www.ecotel.co.za/
🔍 處理中 [427/140978]: http://www.optus.com.au/shop/support/answer?answerId=1842&question=%20&typeId=6
🔍 處理中 [441/140978]: https://www.tumblr.com/safe-mode?url=https%3A%2F%2Fopiumudkr.tumblr.com%2F
🔍 處理中 [451/140978]: http://www.maestragemma.com/
🔍 處理中 [449/140978]: http://iceage.wikia.com/wiki/Ice_Age:_Collision_Course
🔍 處理中 [439/14

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...
🔍 處理中 [1601/140978]: https://en.wiktionary.org/wiki/provenance
🔍 處理中 [1604/140978]: http://pawanhospital.co.in/fargo/login.php
🔍 處理中 [1621/140978]: https://www.quora.com/In-statistics-what-is-a-type-1-and-type-2-error
🔍 處理中 [1610/140978]: https://www.facebook.com/ProjectProvenance
🔍 處理中 [1608/140978]: https://en.wikipedia.org/wiki/Open_Source_Hardware_Association
🔍 處理中 [1624/140978]: https://norrjevesfi.files.wordpress.com/2015/11/huawei-mobile-broadband-e176-manual.pdf
🔍 處理中 [1623/140978]: http://www.schrockguide.net/
🔍 處理中 [1635/140978]: https://www.vanishingincmagic.com/magic/close-up-magic/
🔍 處理中 [1637/140978]: https://simple.wikipedia.org/wiki/Compact_disc
🔍 處理中 [1636/140978]: http://extremalpro.com.ua
🔍 處理中 [1649/140978]: https://en.wikipedia.org/wiki/Baptismal_font
🔍 處理中 [1647/140978]: https://www.shortlink.net/VERIFY-ATM
🔍 處理中 [1642/140978]: http://www.calculatorsoup.com/calculators/math/index.php
🔍 處理中 [1654/140978]: http://dmcritchie.mvps.or

...

/tmp/ipykernel_1531/3978108832.py:49: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(html_content, 'lxml')


🔍 處理中 [92588/140978]: https://lisellapropertygroup.com/poste/ltaliane-13-02-2023/
🔍 處理中 [92589/140978]: https://giii-72b1f.web.app/
🔍 處理中 [92592/140978]: https://www.anexia.es
🔍 處理中 [92590/140978]: https://hw9q8pinxf.firebaseapp.com/
🔍 處理中 [92593/140978]: https://steamcommuity.co.in/profiies/76526073959498373
🔍 處理中 [92594/140978]: http://www.manhphu.xyz
🔍 處理中 [92598/140978]: https://www.tipperlinne.com

💾 --- 已處理至 92600 筆，寫入 Google Drive 中... ---
♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...
🔍 處理中 [92603/140978]: https://muffleddishonesttranslation.falco740.repl.co/
🔍 處理中 [92604/140978]: http://cargoeasytracker.com/
🔍 處理中 [92602/140978]: http://www.authorizeddns.us
🔍 處理中 [92607/140978]: https://www.palladium-megaverse.com
🔍 處理中 [92606/140978]: https://smgolamalif.github.io/facebook-login-page
🔍 處理中 [92613/140978]: http://librerialevalle.eshost.com.ar/load.php
🔍 處理中 [92609/140978]: https://maildinshaakckjnw200.web.app/
🔍 處理中 [92620/140978]: http://www.adlransna.ml
🔍 處理中 [92612/140978]: https://cf-

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


🔍 處理中 [92802/140978]: http://www.amerioaoexpreos.co.jp.epgajsk.cn/
🔍 處理中 [92805/140978]: http://www.lingnuoguanjia.com
🔍 處理中 [92804/140978]: http://wvw.lmterbank.benefitpe.com
🔍 處理中 [92813/140978]: https://urbanexcursionspdx.com/wp-admin/maint/coxxa/web/login.php?web/cox/signon#/now/overviewaccounts/overview/index=st2uguarbhmpnlxqxx5teqynbybfx18gjtss7ad93rmc0xjabxoamhkgqos5qxavmooaf6j3cx0vlvxw
🔍 處理中 [92817/140978]: http://www.x6.xwmpn.com
🔍 處理中 [92816/140978]: https://is.gd/controllo_sicurezza_banking
🔍 處理中 [92806/140978]: https://www.niordc.ir
🔍 處理中 [92812/140978]: http://www.bestsexologist.xyz
🔍 處理中 [92807/140978]: https://chrono-posts-team-4.adalo.com/chrono-post?_gl=1%2a1e145s4%2a_ga%2amtqymdqwodawoc4xnjcxnjcwmdm3%2a_ga_swt45dv35l%2amty3mty3mda0mc4xljeumty3mty3mda3mc4zmc4wlja.&amp;target=998942e43ce445e7a8ed43cc51862f42&amp;params=%7b%7d
🔍 處理中 [92814/140978]: https://xn--visualizandonline-maro-k7b.com/luiza/?userid=3&amp;uri=awbtwecpiqv8t7r6jnlhtig4axtxde9f+wqveq6eqgy=
🔍 處理中 [92818

Exception: Browser.close: Connection closed while reading from the driver

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
FILE_NAME = "/content/drive/MyDrive/畢業專題資料及/phishing_dataset_html_combine2_html.csv"
df = pd.read_csv(FILE_NAME)
print(len(df[df['feature_extracted'] == 1]))
print(len(df[(df['feature_extracted'] == 1) & (df['target'] == 0)]))
print(len(df[df['feature_extracted'] == 0]))

Mounted at /content/drive
82153
13415
58825


# **這個是捨棄使用aiohttp，避免爬蟲被檔的版本**

In [ ]:
import pandas as pd
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import re
import asyncio
import os
import numpy as np
import gc
import ipaddress
from typing import Tuple, Optional
from google.colab import drive

# ====== Playwright Async 模組 ======
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError, Error as PlaywrightError

# --- 📁 第一招：Google Drive 掛載與資料夾設定 ---
drive.mount('/content/drive')

FILE_NAME = "/content/drive/MyDrive/畢業專題Playwright_html/phishing_dataset_html_combine2_html.csv"

# --- ⚙️ 第二招：併發與批次設定 ---
CONCURRENCY_LIMIT = 20   # 同時開啟的分頁數 (Colab 免費版建議 5~10)
BATCH_SIZE = 200        # 每處理幾筆存一次檔
RESTART_INTERVAL = 200  # 每處理幾筆強制重啟瀏覽器釋放記憶體
RENDER_WAIT_TIME = 2

# 建立號誌，限制同時執行的任務數量
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

async def block_agressive_resources(route):
    """阻擋圖片、影片、CSS 載入，大幅提升速度"""
    if route.request.resource_type in ["image", "media", "font", "stylesheet"]:
        await route.abort()
    else:
        await route.continue_()
async def fetch_both_contents_async(page, url: str) -> Tuple[Optional[str], Optional[str]]:
    """使用純 Playwright 一次獲取「靜態原始碼」與「動態渲染 DOM」"""
    html_content = None
    dynamic_content = None

    try:
        # 1. 導向網址，並把 response 存起來
        response = await page.goto(url, timeout=20000, wait_until='domcontentloaded')

        # 2. 🌟 關鍵：立刻從 response 中抽出原始的、未渲染的靜態 HTML (取代 aiohttp)
        if response:
            if response.status >= 400:
                return None, None # 404 等錯誤頁面直接放棄
            html_content = await response.text()

    except PlaywrightError as e:
        if "interrupted by another navigation" in str(e):
            try: await page.wait_for_load_state('domcontentloaded', timeout=15000)
            except: pass
        else:
            return None, None # 其他嚴重連線錯誤直接放棄

    # 3. 等待 JS 渲染與簡單滾動
    await page.wait_for_timeout(RENDER_WAIT_TIME * 1000)
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2);")
    await page.wait_for_timeout(500)

    # 4. 🌟 獲取最終渲染完成的動態 HTML
    dynamic_content = await page.content()

    return html_content, dynamic_content

async def process_single_row(index, row, context, df, html_feature_columns, total_rows):
  """處理單筆資料的工作節點 (Worker)"""
  url = row['url']
  if pd.isna(url) or not str(url).strip():
    df.at[index, 'feature_extracted'] = 0.0
    return

  if not url.startswith('http://') and not url.startswith('https://'):
    url = 'http://' + url

  # 每個任務獨立開一個 Page，避免互相干擾
  page = await context.new_page()
  await page.route("**/*", block_agressive_resources)

  try:
    print(f"🔍 處理中 [{index+1}/{total_rows}]: {url}")

    html_content, dynamic_content = await fetch_both_contents_async(page, url)
    if dynamic_content:
      # 如果 html_content 是 None，就拿 dynamic_content 填補
      html_content = html_content or dynamic_content
    if html_content and dynamic_content:
      # ====== 以下是你的特徵解析邏輯 ======
      # 第三招：統一使用 'lxml' 加速
      soup_static = BeautifulSoup(html_content, 'lxml')
      soup_dynamic = BeautifulSoup(dynamic_content, 'lxml')

      parsed_url = urlparse(url)
      base_domain = parsed_url.netloc.split(':')[0]
      if base_domain.startswith('www.'): base_domain = base_domain[4:]

      # 15 & 16
      meta_tag = soup_static.find('meta', attrs={'http-equiv': lambda x: x and x.lower() == 'refresh'})
      df.at[index, 'has_meta_refresh'] = 1.0 if meta_tag and "url=" in meta_tag.get("content", "").lower() else 0.0
      redirect_kws = ["window.location.href", "location.href", "location.assign", "location.replace"]
      df.at[index, 'has_js_redirect'] = 1.0 if soup_static.find("script", string=lambda s: any(k in s for k in redirect_kws) if s else False) else 0.0

      # 1
      phish_kws = ['login', 'signin', 'account update', 'verify account', 'security alert', 'password', 'bank', 'paypal']
      text_content = soup_dynamic.get_text().lower()
      df.at[index, 'phish_hints'] = 1.0 if any(kw in text_content for kw in phish_kws) else 0.0

      # 2
      domain_parts = base_domain.split('.')
      core_domain = domain_parts[-2] if len(domain_parts) >= 2 and domain_parts[-1] in ['com', 'org', 'net', 'edu', 'gov'] else (domain_parts[-3] if len(domain_parts) >= 3 else domain_parts[0])
      title_tag = soup_dynamic.find('title')
      meta_desc = soup_dynamic.find('meta', attrs={'name': 'description'})
      df.at[index, 'domain_in_brand'] = 1.0 if (title_tag and core_domain in title_tag.get_text().lower()) or (meta_desc and core_domain in meta_desc.get('content', '').lower()) else 0.0

      # 3, 4, 5, 6, 7, 11
      all_links = soup_dynamic.find_all('a', href=True)
      df.at[index, 'nb_hyperlinks'] = len(all_links)

      int_links, ext_links, red_count, err_count = 0, 0, 0, 0
      is_safe_anchor = 1.0
      suspicious_kws = ['bit.ly', 'tinyurl', 'goo.gl', 't.co']

      for link in all_links:
          href = link['href']
          if href.startswith('#'): continue
          full_url = urljoin(url, href)
          linked_domain = urlparse(full_url).netloc
          if linked_domain == parsed_url.netloc:
            int_links += 1
          else:
            ext_links += 1
            if link.get('onclick') and 'window.location' in link.get('onclick', ''): red_count += 1
            elif link.get('target') == '_blank' and 'redirect' in link.get_text().lower(): red_count += 1
            if 'error' in full_url.lower() or '404' in full_url or not linked_domain: err_count += 1

            try: ipaddress.ip_address(linked_domain); is_safe_anchor = 0.0
            except ValueError: pass
            if any(kw in linked_domain.lower() for kw in suspicious_kws): is_safe_anchor = 0.0

      total_links = int_links + ext_links
      df.at[index, 'ratio_intHyperlinks'] = int_links / total_links if total_links > 0 else 0.0
      df.at[index, 'ratio_extHyperlinks'] = ext_links / total_links if total_links > 0 else 0.0
      df.at[index, 'ratio_extRedirection'] = red_count / len(all_links) if all_links else 0.0
      df.at[index, 'ratio_extErrors'] = err_count / len(all_links) if all_links else 0.0
      df.at[index, 'safe_anchor'] = is_safe_anchor

      # 8, 9, 10
      favicon = soup_dynamic.find('link', rel=lambda x: x and 'icon' in x.lower())
      df.at[index, 'external_favicon'] = 1.0 if favicon and 'href' in favicon.attrs and urlparse(urljoin(url, favicon['href'])).netloc != parsed_url.netloc else 0.0
      df.at[index, 'links_in_tags'] = sum(1 for tag in soup_dynamic.find_all(['a', 'script', 'img', 'link', 'iframe', 'form']) if 'href' in tag.attrs or 'src' in tag.attrs or (tag.name == 'form' and 'action' in tag.attrs))

      media_tags = soup_dynamic.find_all(['img', 'audio', 'video', 'source'])
      ext_media = sum(1 for tag in media_tags if (tag.get('src') or tag.get('href')) and urlparse(urljoin(url, tag.get('src') or tag.get('href'))).netloc != parsed_url.netloc)
      df.at[index, 'ratio_extMedia'] = ext_media / len(media_tags) if media_tags else 0.0

      # 12, 13, 14
      df.at[index, 'empty_title'] = 1.0 if not (title_tag and title_tag.string and title_tag.string.strip()) else 0.0
      df.at[index, 'domain_in_title'] = 1.0 if title_tag and title_tag.string and base_domain in title_tag.string.lower() else 0.0
      copyright_tags = soup_dynamic.find_all(text=re.compile(r'©|copyright|all rights reserved', re.IGNORECASE))
      df.at[index, 'domain_with_copyright'] = 1.0 if any(base_domain in tag.lower() for tag in copyright_tags) else 0.0

      df.at[index, 'feature_extracted'] = 1.0

    else:
      for col in html_feature_columns: df.at[index, col] = 0.0
      df.at[index, 'feature_extracted'] = 0.0

  except Exception as e:
    for col in html_feature_columns: df.at[index, col] = 0.0
    df.at[index, 'feature_extracted'] = 0.0
  finally:
    # ✅ 確保用完的 Page 一定會被關閉，釋放記憶體！
    await page.close()
async def safe_worker_wrapper(index, row, context, df, html_feature_columns, total_rows):
    # 先拿到「並發許可證（入座）」，才開始算 90 秒！
    async with semaphore:
        try:
            # 入座後，給他 90 秒的時間吃拉麵（爬蟲）
            await asyncio.wait_for(
                process_single_row(index, row, context, df, html_feature_columns, total_rows),
                timeout=90.0
            )
        except asyncio.TimeoutError:
            print(f"⏰ [上帝大限] 第 {index+1} 筆 URL ({row['url']}) 卡死超過 90 秒，已強制拔管！")
            df.at[index, 'feature_extracted'] = 0.0
            for col in html_feature_columns:
                if col != 'feature_extracted':
                    df.at[index, col] = 0.0
        except Exception as e:
            print(f"❌ [未預期崩潰] 第 {index+1} 筆 URL 發生錯誤: {e}")
            df.at[index, 'feature_extracted'] = 0.0

async def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
    html_feature_columns = [
        'phish_hints', 'domain_in_brand', 'nb_hyperlinks', 'ratio_intHyperlinks',
        'ratio_extHyperlinks', 'ratio_extRedirection', 'ratio_extErrors',
        'external_favicon', 'links_in_tags', 'ratio_extMedia', 'safe_anchor',
        'empty_title', 'domain_in_title', 'domain_with_copyright',
        'has_meta_refresh', 'has_js_redirect', 'feature_extracted'
    ]

    if 'feature_extracted' not in df.columns:
        df[html_feature_columns] = np.nan

    total_rows = len(df)
    print(f"總共 {total_rows} 筆資料準備進入併發處理...")

    async with async_playwright() as p:
      browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage'])
      context = await browser.new_context(ignore_https_errors=True)
      try:
        # 🔪 將任務切成 BATCH_SIZE 大小的批次
        for i in range(0, total_rows, BATCH_SIZE):
          batch_df = df.iloc[i:i+BATCH_SIZE]
          tasks = []

          for index, row in batch_df.iterrows():
            # 跳過已經成功提取特徵的資料 (斷點續傳)
            if index < 120000:continue
            if row.get('feature_extracted') == 1.0:continue
            # 建立併發任務
            task = asyncio.create_task(
                safe_worker_wrapper(index, row, context, df, html_feature_columns, total_rows)
            )
            tasks.append(task)

            # 如果這個批次有任務需要跑，就一口氣執行它們
          if tasks:await asyncio.gather(*tasks, return_exceptions=True)
          else:continue
          # 批次結束，進行存檔
          print(f"\n💾 --- 已處理至 {min(i+BATCH_SIZE, total_rows)} 筆，寫入 Google Drive 中... ---")
          df.to_csv(FILE_NAME, index=False)

          # 記憶體回收機制
          if i > 0 and i % RESTART_INTERVAL == 0:
            print("♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...")
            try: await context.close()
            except: pass
            try: await browser.close()
            except: pass
            gc.collect()
            browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage'])
            context = await browser.new_context(ignore_https_errors=True)

      except (KeyboardInterrupt, asyncio.CancelledError):
        print("\n🛑 偵測到手動中斷，儲存最終進度...")
        df.to_csv(FILE_NAME, index=False)
      finally:
        await browser.close()
    return df

async def main():
    os.system("pkill -9 -f chrome")

    if os.path.exists(FILE_NAME):
        print(f"✅ 找到 Drive 中的進度檔: {FILE_NAME}")
        df = pd.read_csv(FILE_NAME)
        # 如果你想「全部重來」，把下面這行取消註解：
        # df['feature_extracted'] = np.nan
    else:
        print(f"⚠️ 找不到進度檔。請確保原始 CSV 存在，或者修改讀取路徑。")
        # 這裡請替換成你的「最原始資料」的讀取路徑
        # df = pd.read_csv("/content/原本的原始檔.csv")
        return

    df_updated = await process_dataset(df)
    print("🎉 任務徹底完成！")

# 啟動任務
await main()

Mounted at /content/drive
✅ 找到 Drive 中的進度檔: /content/drive/MyDrive/畢業專題Playwright_html/phishing_dataset_html_combine2_html.csv
總共 140978 筆資料準備進入併發處理...
🔍 處理中 [120008/140978]: https://www.rachelfoxfrazier.com
🔍 處理中 [120001/140978]: http://www.mitid-approve.20-203-169-150.cprapid.com/nord1/login.php
🔍 處理中 [120005/140978]: http://www.omgwowxisr.ga
🔍 處理中 [120002/140978]: https://account-billing.emmausadventures.com/
🔍 處理中 [120010/140978]: https://zibat100.ir/goe/auth/
🔍 處理中 [120011/140978]: https://proud-moon-31d2.kosaci4411.workers.dev/
🔍 處理中 [120012/140978]: https://www.smallbutdigital.com
🔍 處理中 [120025/140978]: https://mail.groupwayciv0t6.mmk.work.gd/vhsfhqpdhdsih6
🔍 處理中 [120021/140978]: http://www.doltox.fr/
🔍 處理中 [120029/140978]: https://215b2e67-7dc1-4a96-a74b-e64b990f203d.id.repl.co
🔍 處理中 [120022/140978]: https://maildinshaakckjnw49.firebaseapp.com/
🔍 處理中 [120028/140978]: https://www.aboutsucculents.com
🔍 處理中 [120030/140978]: https://gateway.pinata.cloud/ipfs/bafybeihxpivdzu4eno4oe7

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


🔍 處理中 [120401/140978]: https://sukoonphysicaltherapy.com/wp-content/plugins/usagedd/apib1anz.comapinetbank.htm
🔍 處理中 [120408/140978]: https://storage.yandexcloud.net/auth-login-ioo5jdgc2ttuqnttt9pwaigtiqiw6h1k8j041qdojaqv8/index.html
🔍 處理中 [120416/140978]: http://www.chubsmail.com
🔍 處理中 [120414/140978]: https://maildinshaakckjnw105.web.app/
🔍 處理中 [120415/140978]: https://storageapi.fleek.co/f93bbabc-c9fe-416b-9895-abc3c414d6f4-bucket/tec4/index.html
🔍 處理中 [120418/140978]: https://1c2f6d45-455e-4661-bf6a-0a41b6ed69ab.id.repl.co/
🔍 處理中 [120426/140978]: https://www.theindependentbd.com
🔍 處理中 [120429/140978]: https://slatt-103257.weeblysite.com/
🔍 處理中 [120430/140978]: http://virkunarzr.gq/updation%202/
🔍 處理中 [120431/140978]: http://auth-sso-biglobe-sso-mail-63e97b64649af.esup.edu.br/auth-api-webmail-1-sso-rui-index/login.html?resource_url=https:/auth.sso.biglobe.ne.jp/webmail=259913&session=101212
🔍 處理中 [120432/140978]: https://www.rebelnews.com
🔍 處理中 [120433/140978]: http://teyop81972.tem

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...
🔍 處理中 [120803/140978]: https://www.baodaknong.org.vn
🔍 處理中 [120804/140978]: https://novo-sicredi-black.com/?page=home&amp;amp;a=208a545d9c9920e5b82164095ef319bad2175dc5secure2158133160d9d280afd9baa68ac7d4411e4dfa5aaccess&amp;amp;b=c307f85ef63c038db5451a1042f6f71035959e14secureb88a92fff52f9bef397d01a21ebe53836dcf62fbaccess&amp;amp;c=4b5a9afcee2770f60fda9be064592b20a861a360secure9c2abcaa953117131aa7425287bb7855c77d94e9access
🔍 處理中 [120807/140978]: https://view.genial.ly/63617982a9d9a400175c5e69
🔍 處理中 [120808/140978]: http://www.f0559435.xsph.ru
🔍 處理中 [120812/140978]: http://www.kueronekayaeotn.co.jp.kuerocekayaaoto.atwtfm.top/ai/?authenticated=true&amp;amp;amp;amp;openid/gp/signin/x&amp;amp;amp;amp;i=a&amp;amp;amp;amp;oauth=m&amp;amp;amp;amp;i?ie=utf8&amp;amp;amp;amp;ref_=rhf_custrec_signin33c720b293c50b5b226d772bf10b26d4acad8a4f
🔍 處理中 [120817/140978]: https://attcom-107046.square.site/
🔍 處理中 [120809/140978]: https://www.uploadkon.ir
🔍 處理中 [120819/14097

...

♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...
🔍 處理中 [139207/140978]: https://mdzxpodz.com/bertobos
🔍 處理中 [139204/140978]: http://www.meinvcaita.com
🔍 處理中 [139208/140978]: https://s.id/1g51e
🔍 處理中 [139211/140978]: https://interbankgt.htmlserve001.repl.co/desk/
🔍 處理中 [139209/140978]: https://help-id-meta-354262888.web.app/
🔍 處理中 [139216/140978]: http://www.ws029-proxy-ajcryptominer.ajplugins.com
🔍 處理中 [139212/140978]: https://leywino.github.io/discord-responsive
🔍 處理中 [139214/140978]: https://3uml42q7h5iwui7w6za2po5gwxuhzah3i7ywuunmnlle6cye-ipfs-w3s-link.translate.goog/01.html?test@example.com+&_x_tr_hp=bafybeigzpw&_x_tr_sl=auto&_x_tr_tl=en-gb&_x_tr_hl=en-gb
🔍 處理中 [139230/140978]: https://www.didierbabarit-photographe.com
🔍 處理中 [139217/140978]: https://gateway.pinata.cloud/ipfs/bafybeieejhl3a4adbxorcjdmaiejtskbbgkdgsdlidc2igre64pphfcwta
🔍 處理中 [139218/140978]: http://att-100689-107113.square.site/
🔍 處理中 [139231/140978]: https://rough-feather-ca25.51g0jm50281.workers.dev/
🔍 處理中 [139219/140978]: https

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


🔍 處理中 [139401/140978]: http://www.lingx.club
🔍 處理中 [139402/140978]: https://fb-restriction-case-5e7cd.web.app/
🔍 處理中 [139404/140978]: https://www.fundsurfer.com
🔍 處理中 [139405/140978]: https://www.floodgateacademy.com
🔍 處理中 [139406/140978]: http://jkgscjbjasc.square.site/
🔍 處理中 [139408/140978]: https://www.acepnow.com
🔍 處理中 [139415/140978]: https://twtr.to/ooae
🔍 處理中 [139410/140978]: https://www.mesharepoint.com/eur/db6905d7-04a1-4f87-ac15-7e6ad415f416/f9a3c1ce-492a-4c5d-b40f-2dca424e2a97/6203294f-aaa3-408b-8456-9274c48de812/login?id=qxnfohn0nvdpc055shk3a3fjbw10qnrkbxa0v0xibmhiy29hceizzhlxtc8yvzbbv0rxykjdbtjyqu90ognwsuxtmu9ralm5zytnqmhraw8yq2qwckzzmxfxckq3bmxkuit1a2zswg5ydtzdyjjyrfv3agrezedurefhagivl0rnemn0eeverwjdmlqxskfmdmj0culir09wttezvky4cxb0ww50tkdssdbzngf3wu9hzfdxc0n5ejdeaxptbjfbq0lrrlpwvelyc2farzhmzfz3dwj0dmfxtxvfuhezzfqwynhxk3hibufxogtdwfaxugo0twfsqwu5azluvgdxsvq1zezhy0domk5jbnrlewqvr2ljbtm0vtc1bdzpqlhjm2jzevlqsgd4l0vyekhybmvguvpptjj1k0hxyjdjmenrq0luogowz2ljwlavudnbk3fntg55zmw3d

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


🔍 處理中 [139805/140978]: https://docs.google.com/forms/d/e/1faipqlsfrw9nsla2lpenbntfmz3q_bu-6vayjudjntqikzcei2rl3ya/viewform?usp=send_form
🔍 處理中 [139808/140978]: http://mingovpllkno.ditapersne.ml/
🔍 處理中 [139814/140978]: https://segundavlaonline.com/?gclid=eaiaiqobchmir9ztpykl_qivgt-rch2iygukeaayasaaegl36_d_bwe
🔍 處理中 [139809/140978]: https://www.anime-recorder.com
🔍 處理中 [139815/140978]: http://www.likeafish.oss-us-east-1.aliyuncs.com
🔍 處理中 [139812/140978]: https://verification-4.web.app/
🔍 處理中 [139816/140978]: https://docs.google.com/forms/d/e/1faipqlscjsewetq_v_cbpyh9dzjwsaoldxz2zx816022wp1vej8gv1g/viewform
🔍 處理中 [139819/140978]: https://www.rvapc.com
🔍 處理中 [139820/140978]: https://btinternet-106850.weeblysite.com/
🔍 處理中 [139821/140978]: https://gateway.pinata.cloud/ipfs/bafybeicivf4lssd2v6zxbcgjv37mwulh2u4t4fquswv53omeunyzq6yaku/
🔍 處理中 [139822/140978]: https://bt-107125.square.site/
🔍 處理中 [139823/140978]: https://bafybeicut2gtz2rcrdw2vby4vlndj2ehe7ph64scqtjwjqw6kcpy6esdti.ipfs.cf-ipfs.c